# Проверка нерасчетных атрибутов по agr_id и INN

Тетрадка достает нерасчетные атрибуты напрямую из ODS/Alpha без витрины DAG.

Поля:
- cdi_id
- inn
- ogrn
- client_name_from_companies
- client_name_from_merchants
- mcc_code
- contract_number
- contract_start_dt
- contract_end_dt
- filial_rf
- vsp_name
- vsp_code
- tariff_name

Оптимизация по памяти:
- ранняя фильтрация по одному agr_id
- узкие join по ключам
- без trx-таблиц на этом шаге


In [ ]:
import pandas as pd

agr_id = "104773604399"
target_inn = "401400143905"

# Параметры расчетного блока (месяц проверки)
month_start = "2026-02-01"
month_end = "2026-02-29"

# Бизнес-константа для АУР
aur_rate = 1926


In [ ]:
from rail_connectors.connection import connect
from connection_secrets import LAKE_USER, LAKE_PASSWORD

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={
        'user_name': LAKE_USER,
        'password': LAKE_PASSWORD,
    }
)

imp._init_connection()
print('Impala connection initialized')

In [ ]:
sql = f"""
with
params as (
    select
        cast('{agr_id}' as string) as agr_id_str,
        '{target_inn}' as target_inn
),

agr_one as (
    select
        a.abs_agr_id,
        a.n_agr,
        a.c_agr_number,
        a.n_cmp_client,
        a.d_valid_from,
        a.d_valid_to
    from ods_alpha.scd1_agreements a
    join params p
      on cast(a.abs_agr_id as string) = p.agr_id_str
    where a.acq_class = 'SA'
      and a.abs_agr_id is not null
),

cmp_one as (
    select
        c.n_cmp,
        c.c_inn,
        c.c_cmp_name
    from ods_alpha.scd1_companies c
    join agr_one a
      on a.n_cmp_client = c.n_cmp
    join params p
      on c.c_inn = p.target_inn
),

r2_one as (
    select
        m.id as agr_id,
        m.c_cl_org,
        m.c_depart,
        m.c_tariff_plan
    from ods.scd1_z_r2_ip_merchants m
    join agr_one a
      on cast(m.id as string) = cast(a.abs_agr_id as string)
),

dep_one as (
    select
        d.id,
        d.c_num as vsp_code,
        d.c_name as vsp_name,
        d.c_filial
    from ods.scd1_z_depart d
    join r2_one r
      on r.c_depart = d.id
),

br_one as (
    select
        b.id,
        b.c_shortlabel as filial_rf
    from ods.scd1_z_branch b
    join dep_one d
      on d.c_filial = b.id
),

corp_one as (
    select
        z.id,
        z.c_register_gos_reg_num_rec as ogrn
    from ods.scd1_z_cl_corp z
    join r2_one r
      on r.c_cl_org = z.id
),

tariff_one as (
    select
        t.id,
        t.c_name as tariff_name
    from ods.scd1_z_r2_tariff_plan t
    join r2_one r
      on r.c_tariff_plan = t.id
),

alpha_merch_small as (
    select
        m.n_cmp,
        max(m.n_mcc) as mcc_code,
        max(m.c_mrc_name) as client_name_from_merchants
    from ods_alpha.scd1_merchants m
    join agr_one a
      on a.n_cmp_client = m.n_cmp
    group by m.n_cmp
),

ocrm_ssp_ranked as (
    select
        cast(se.x_inn as string) as inn,
        cast(se.x_area_resp as string) as ssp,
        count(distinct cast(se.x_area_resp as string)) over (
            partition by cast(se.x_inn as string)
        ) as ssp_variants_cnt,
        row_number() over (
            partition by cast(se.x_inn as string)
            order by cast(se.last_upd as timestamp) desc,
                     cast(se.created as timestamp) desc,
                     se.row_id desc
        ) as rn
    from ocrm_ul.s_org_ext se
    join cmp_one c
      on cast(c.c_inn as string) = cast(se.x_inn as string)
    where se.x_inn is not null
      and se.x_area_resp is not null
      and trim(cast(se.x_area_resp as string)) <> ''
),

ocrm_ssp_one as (
    select
        inn,
        ssp,
        ssp_variants_cnt
    from ocrm_ssp_ranked
    where rn = 1
)

select
    cast(a.abs_agr_id as string) as cdi_id,
    c.c_inn as inn,
    co.ogrn as ogrn,
    c.c_cmp_name as client_name_from_companies,
    am.client_name_from_merchants,
    am.mcc_code,
    a.c_agr_number as contract_number,
    a.d_valid_from as contract_start_dt,
    a.d_valid_to as contract_end_dt,
    b.filial_rf,
    d.vsp_name,
    d.vsp_code,
    t.tariff_name,
    ocrm.ssp,
    ocrm.ssp_variants_cnt
from agr_one a
join cmp_one c on c.n_cmp = a.n_cmp_client
left join r2_one r on cast(r.agr_id as string) = cast(a.abs_agr_id as string)
left join dep_one d on d.id = r.c_depart
left join br_one b on b.id = d.c_filial
left join corp_one co on co.id = r.c_cl_org
left join tariff_one t on t.id = r.c_tariff_plan
left join alpha_merch_small am on am.n_cmp = a.n_cmp_client
left join ocrm_ssp_one ocrm on ocrm.inn = cast(c.c_inn as string)
;
"""

with imp:
    df_non_calc = imp.fetch(sql)

display(df_non_calc)


In [ ]:
# ЭТАП 1: таблица с нерасчетными показателями
result_stage1 = df_non_calc.copy()
display(result_stage1)

In [ ]:
# ЭТАП 2: расчетные показатели за месяц (точки, терминалы, операции, сумма операций, АУР)
sql_calc = f"""
with
params as (
    select
        cast('{agr_id}' as string) as agr_id_str,
        '{target_inn}' as target_inn,
        cast('{month_start}' as date) as month_start,
        cast('{month_end}' as date) as month_end,
        cast({aur_rate} as decimal(18,2)) as aur_rate
),

agr_one as (
    select
        a.abs_agr_id,
        a.n_agr,
        a.n_cmp_client
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    join params p
      on cast(a.abs_agr_id as string) = p.agr_id_str
     and c.c_inn = p.target_inn
    where a.acq_class = 'SA'
      and a.abs_agr_id is not null
),

retl_scope as (
    select
        a.abs_agr_id as agr_id,
        m.c_nmrc as retl_id
    from agr_one a
    join ods_alpha.scd1_merchants m
      on m.n_cmp = a.n_cmp_client
    where m.c_nmrc is not null
),

term_scope as (
    select
        r.agr_id,
        r.retl_id,
        t.c_nter as term_id,
        cast(t.d_ter_install as date) as d_ter_install,
        cast(coalesce(t.d_ter_close, '2999-12-31') as date) as d_ter_close
    from retl_scope r
    join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = r.retl_id
    where t.c_nter is not null
),

retl_term_month as (
    select
        ts.agr_id,
        count(distinct ts.retl_id) as cnt_retl_feb,
        count(distinct ts.term_id) as cnt_term_feb
    from term_scope ts
    join params p on 1=1
    where ts.d_ter_install <= p.month_end
      and ts.d_ter_close >= p.month_start
    group by ts.agr_id
),

trx_base as (
    select
        t.n_trx,
        t.n_amt_src
    from ods_alpha.scd1_trx t
    join params p on 1=1
    where t.c_trx_class = 'SA'
      and t.c_trx_type = 'S01'
      and t.c_nter is not null
      and t.ods_deleted_flg <> '1'
      and t.cf_trx_stat <> 'R'
      and to_date(cast(t.d_trx_orig as timestamp)) between p.month_start and p.month_end
),

trx_join as (
    select
        a.abs_agr_id as agr_id,
        b.n_trx,
        b.n_amt_src
    from trx_base b
    join ods_alpha.scd1_trx_acq ta
      on ta.n_trx = b.n_trx
    join agr_one a
      on a.n_agr = ta.n_agr
),

trx_agg as (
    select
        agr_id,
        count(distinct n_trx) as operations_cnt_feb,
        sum(n_amt_src) as operations_sum_feb
    from trx_join
    group by agr_id
)

select
    cast(a.abs_agr_id as string) as cdi_id,
    coalesce(rt.cnt_retl_feb, 0) as cnt_retl_feb,
    coalesce(rt.cnt_term_feb, 0) as cnt_term_feb,
    coalesce(tx.operations_cnt_feb, 0) as operations_cnt_feb,
    coalesce(tx.operations_sum_feb, 0) as operations_sum_feb,
    cast(coalesce(rt.cnt_retl_feb, 0) as decimal(18,2)) * p.aur_rate as aur_feb
from agr_one a
join params p on 1=1
left join retl_term_month rt
  on rt.agr_id = a.abs_agr_id
left join trx_agg tx
  on tx.agr_id = a.abs_agr_id
;
"""

with imp:
    df_calc = imp.fetch(sql_calc)

display(df_calc)

In [ ]:
# ЭТАП 3: дополняем таблицу расчетными показателями
calc_cols = [
    "cdi_id",
    "cnt_retl_feb",
    "cnt_term_feb",
    "operations_cnt_feb",
    "operations_sum_feb",
    "aur_feb",
]

result_stage2 = result_stage1.merge(
    df_calc[calc_cols],
    on="cdi_id",
    how="left",
)

display(result_stage2)